### Forecasting Noise Patterns

Now for the moment of truth. We’re taking our model and using three years of historical data (2021–2024) to predict how noise complaint variability shifts across different NYC PUMAs.

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API",
    category=UserWarning,
)

In [2]:
import random
from datetime import date, timedelta
import pymc as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import geopandas as gpd
from keplergl import KeplerGl
from helpers import ( 
                    prep_the_data, 
                      export_puma_kepler, 
                      make_daily_table_for_model_with_nta,
                        make_typical_week_2025,
                      make_daily_observed_2025,
                      load_idata,
                      compare_models_loo_waic,
                      kepler_typical_week_from_dow_complaint,
                      score_against_real_2025_days,
                      crps_from_draws, 
                      elpd_from_draws,
                      random_summer_date,
                      summarize_forecast_metrics,
                      forest_day_puma_intervals,
                      plot_puma_day_interval,
                      plot_coverage_curve,
                      normalize_summary_for_comparison,
                      rebuild_daily_cmp_2025_model3,
                      summarize_model_performance
                    )




### Load + prepare data: 2021-2024 

In [3]:
df_puma = pd.read_parquet(
    "../data/processed/features/puma_noise_counts.parquet"
)


In [4]:
df_puma = prep_the_data(df_puma)

In [5]:
df_puma_2021__2024 = df_puma.loc[
    (df_puma["created_bucket"] >= "2021-01-01") &
    (df_puma["created_bucket"] <  "2024-12-31")
].copy()


In [6]:
df_puma_2021__2024.head()

,puma,complaint_type,descriptor,location_type,created_bucket,time_of_day,complaint_count,nta,area_share_of_puma,nta_name,nta_puma,dow,month,is_weekend,month_year,descriptor_group,dow_complaint
0,4103,Noise,Noise,NR5,2022-06-09,day,1,MN0303,0.403905,East Village,East Village — 4103,Thursday,June,0,June__2022,Other,OTHER__Thursday
1,4103,Noise,"Noise, Barking Dog",NaN,2021-07-10,night,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,July,1,July__2021,Animal,ANIMAL__Saturday
2,4103,Noise,"Noise, Barking Dog",NaN,2021-08-07,day,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,August,1,August__2021,Animal,ANIMAL__Saturday
3,4103,Noise,"Noise, Barking Dog",NaN,2021-08-13,day,1,MN0303,0.403905,East Village,East Village — 4103,Friday,August,0,August__2021,Animal,ANIMAL__Friday
4,4103,Noise,"Noise, Barking Dog",NaN,2021-08-15,night,2,MN0303,0.403905,East Village,East Village — 4103,Sunday,August,1,August__2021,Animal,ANIMAL__Sunday


#### We’re going to focus on Social/Party complaints, since they consistently show up as the largest share of noise complaints in the data.

In [7]:
COMPLAINT = "Social / Party"
# COMPLAINT = "Animal"
# COMPLAINT = "Construction / Industrial"
# COMPLAINT = "Mechanical Equipment"

# Rebuild daily_df_train exactly as in training
daily_df_train, coords = make_daily_table_for_model_with_nta(
    df_puma_2021__2024,
    complaint_value=COMPLAINT,
)



In [8]:
# Rebuild puma → nta mapping
puma_nta_map = (
    daily_df_train[["puma_idx", "nta_idx"]]
    .drop_duplicates()
    .sort_values("puma_idx")
)


In [9]:

puma_to_nta_idx = puma_nta_map["nta_idx"].to_numpy()


### Load the previous model.

In [10]:
idata_puma_nta_pois = load_idata("../data/processed/models/model2_puma_nta_idata.nc")

✅ Loaded idata <- ../data/processed/models/model2_puma_nta_idata.nc


WE begin by building the posterior 

In [11]:
lam_post = idata_puma_nta_pois.posterior["lam"]

# Posterior mean forecast
df_forecast = (
    lam_post
    .mean(dim=("chain", "draw"))
    .to_dataframe(name="lam_forecast")
    .reset_index()
)


We calculate the 90% HDI (Highest Density Interval) to capture the plausible range 
of expected noise complaints for each PUMA and day of week.

In practical terms, this gives us the lower and upper bounds — the minimum and maximum complaint 
counts we’d reasonably expect, given the posterior distribution.

In [12]:
hdi = az.hdi(lam_post, hdi_prob=0.9)["lam"]

hdi_df = (
    hdi.to_dataframe(name="lam_hdi")
       .reset_index()
       .pivot_table(
           index=["puma", "dow"],
           columns="hdi",
           values="lam_hdi",
       )
       .reset_index()
       .rename(columns={"lower": "lam_low_90", "higher": "lam_high_90"})
)

df_forecast = df_forecast.merge(hdi_df, on=["puma", "dow"], how="left")
df_forecast["lam_width_90"] = (
    df_forecast["lam_high_90"] - df_forecast["lam_low_90"]
)


### Load Complaints from the summer of 2025

In [13]:
df_puma_311_2025 = df_puma.loc[
    (df_puma["created_bucket"] >= "2025-01-01") 
    & (df_puma["descriptor_group"] ==  COMPLAINT)
#    & (df_puma["time_of_day"] ==  "night")
].copy()


In [14]:
df_puma_311_2025.head()

,puma,complaint_type,descriptor,location_type,created_bucket,time_of_day,complaint_count,nta,area_share_of_puma,nta_name,nta_puma,dow,month,is_weekend,month_year,descriptor_group,dow_complaint
958,4103,Noise - Commercial,Car/Truck Music,Store/Commercial,2025-07-12,night,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,July,1,July__2025,Social / Party,SOCIAL_PARTY__Saturday
959,4103,Noise - Commercial,Car/Truck Music,Store/Commercial,2025-07-15,night,1,MN0303,0.403905,East Village,East Village — 4103,Tuesday,July,0,July__2025,Social / Party,SOCIAL_PARTY__Tuesday
960,4103,Noise - Commercial,Car/Truck Music,Store/Commercial,2025-08-01,day,1,MN0303,0.403905,East Village,East Village — 4103,Friday,August,0,August__2025,Social / Party,SOCIAL_PARTY__Friday
961,4103,Noise - Commercial,Car/Truck Music,Store/Commercial,2025-08-11,night,1,MN0303,0.403905,East Village,East Village — 4103,Monday,August,0,August__2025,Social / Party,SOCIAL_PARTY__Monday
962,4103,Noise - Commercial,Car/Truck Music,Store/Commercial,2025-08-16,day,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,August,1,August__2025,Social / Party,SOCIAL_PARTY__Saturday


### Aggregate noise complaints for a typical week in the summer week per PUMA.

In [15]:
typical_2025 = make_typical_week_2025(
    df_puma_311_2025,
    complaint_col="descriptor_group",
    months = [6, 7, 8]
)


### Compare Forecasted data to Observed 2025 data for a typical week

In [16]:

# -----------------------------
# 0) Daily observed (summer 2025)
# -----------------------------
daily_2025 = make_daily_observed_2025(
    df_puma_311_2025,
    complaint_value=COMPLAINT,          # or None
    complaint_col="descriptor_group",
)



In [17]:
# -----------------------------
# 1) Merge forecast (keyed by puma,dow)
# -----------------------------
forecast = df_forecast_week if "df_forecast_week" in globals() else df_forecast

daily_cmp_2025_model2 = daily_2025.merge(
    forecast,
    on=["puma", "dow"],
    how="left",
)

# -----------------------------
# 2) Point forecast errors
# -----------------------------
daily_cmp_2025_model2["error"] = (
    daily_cmp_2025_model2["daily_count"] - daily_cmp_2025_model2["lam_forecast"]
)
daily_cmp_2025_model2["abs_error"] = daily_cmp_2025_model2["error"].abs()

# Optional: keep lam-interval coverage for reference only.
# IMPORTANT: lam_low_90 / lam_high_90 are intervals for the latent mean rate λ,
# not for realized daily counts. They should NOT be expected to achieve ~90%
# coverage for observed counts.
if {"lam_low_90", "lam_high_90"}.issubset(daily_cmp_2025_model2.columns):
    daily_cmp_2025_model2["within_90_lam"] = (
        (daily_cmp_2025_model2["daily_count"] >= daily_cmp_2025_model2["lam_low_90"]) &
        (daily_cmp_2025_model2["daily_count"] <= daily_cmp_2025_model2["lam_high_90"])
    )
else:
    daily_cmp_2025_model2["within_90_lam"] = pd.NA

# -----------------------------
# 3) Build puma_idx / dow_idx aligned to training coords
# -----------------------------
puma_to_idx = {str(p): i for i, p in enumerate(coords["puma"])}
dow_to_idx  = {str(d): i for i, d in enumerate(coords["dow"])}

daily_cmp_2025_model2["puma_idx"] = daily_cmp_2025_model2["puma"].astype(str).map(puma_to_idx)
daily_cmp_2025_model2["dow_idx"]  = daily_cmp_2025_model2["dow"].astype(str).map(dow_to_idx)

daily_cmp_2025_model2 = daily_cmp_2025_model2.dropna(subset=["puma_idx", "dow_idx"]).copy()
daily_cmp_2025_model2["puma_idx"] = daily_cmp_2025_model2["puma_idx"].astype(int)
daily_cmp_2025_model2["dow_idx"]  = daily_cmp_2025_model2["dow_idx"].astype(int)

# -----------------------------
# 4) Posterior predictive draws for the Poisson hierarchical model
# -----------------------------
lam_draws_m2 = (
    idata_puma_nta_pois.posterior["lam"]
    .stack(sample=("chain", "draw"))
    .transpose("puma", "dow", "sample")
    .values
)  # (n_puma, n_dow, S)

p_idx = daily_cmp_2025_model2["puma_idx"].to_numpy()
d_idx = daily_cmp_2025_model2["dow_idx"].to_numpy()

mu_draws_m2 = lam_draws_m2[p_idx, d_idx, :]  # (n_days, S)
daily_cmp_2025_model2["mu_pred_mean"] = mu_draws_m2.mean(axis=1)

rng = np.random.default_rng(42)
y_pp_model2 = rng.poisson(mu_draws_m2)  # realized daily counts

# -----------------------------
# 5) 90% POSTERIOR PREDICTIVE intervals for counts
# -----------------------------
q_lo, q_hi = 0.05, 0.95
daily_cmp_2025_model2["y_pred_low_90"] = np.quantile(y_pp_model2, q_lo, axis=1)
daily_cmp_2025_model2["y_pred_high_90"] = np.quantile(y_pp_model2, q_hi, axis=1)

daily_cmp_2025_model2["within_90_pred"] = (
    (daily_cmp_2025_model2["daily_count"] >= daily_cmp_2025_model2["y_pred_low_90"]) &
    (daily_cmp_2025_model2["daily_count"] <= daily_cmp_2025_model2["y_pred_high_90"])
)

daily_cmp_2025_model2["pred_width_90"] = (
    daily_cmp_2025_model2["y_pred_high_90"] - daily_cmp_2025_model2["y_pred_low_90"]
)

# Keep y_pp_model2 available for CRPS / ELPD calculations below


We evaluated the model on 385 observations. The Mean Absolute Error (MAE) tells us, on average, how many complaints we’re off by on a typical day. In this case, we’re missing by about 10 complaints per day. The median error is a bit lower — around 6.7 complaints — which suggests most days aren’t wildly off, but there are some larger misses pulling the average up.

So directionally, the model is reasonable. It’s generally pointing the right way. But it’s not especially tight around the true counts.

The bigger issue is the 90% coverage. It’s only about 8%. That means our “90% confidence” interval actually contains the real complaint count just 8% of the time. In plain terms, the uncertainty bands are far too narrow — the model is dramatically underestimating how much variability actually exists.

You can see that in the interval width. The median 90% band is only about 2.7 complaints wide, and on average about 3 complaints. That’s tiny compared to errors of 7–10 complaints. So the model is acting very confident, but the real data are much more volatile.

In short: the point predictions are moderately accurate, but the uncertainty estimates are badly under-calibrated.

In [18]:
# Summary stats (lam-interval-based)
# Predictive summary for Model 2 on realized 2025 daily counts
summary_model2_daily = {
    "N": int(len(daily_cmp_2025_model2)),
    "MAE (lam_forecast)": float(daily_cmp_2025_model2["abs_error"].mean()),
    "Median AE (lam_forecast)": float(daily_cmp_2025_model2["abs_error"].median()),
    "90% Coverage (predictive interval)": float(daily_cmp_2025_model2["within_90_pred"].mean()),
    "Median predictive interval width (90%)": float(daily_cmp_2025_model2["pred_width_90"].median()),
    "Mean predictive interval width (90%)": float(daily_cmp_2025_model2["pred_width_90"].mean()),
    "90% Coverage (lam interval; reference only)": (
        float(pd.to_numeric(daily_cmp_2025_model2["within_90_lam"], errors="coerce").mean())
        if "within_90_lam" in daily_cmp_2025_model2.columns else np.nan
    ),
}

pd.DataFrame(list(summary_model2_daily.items()), columns=["Metric", "Value"])



,Metric,Value
0,N,4913.000000
1,MAE (lam_forecast),15.099076
2,Median AE (lam_forecast),9.354821
3,90% Coverage (predictive interval),0.423774
4,Median predictive interval width (90%),17.000000
5,Mean predictive interval width (90%),18.098026
6,90% Coverage (lam interval; reference only),0.061470


### We are going to visualize the forecasted errors. To see where they are and how off are they.

#### Pick a random target date

In [19]:
target_date = random_summer_date(2025)

target_date

'2025-07-08'

In [20]:
df_one_day_model2    =  daily_cmp_2025_model2[daily_cmp_2025_model2["date"] == pd.Timestamp(target_date)]

In [21]:
gdf_daily_kepler = kepler_typical_week_from_dow_complaint(
    df_one_day_model2,
    puma_geojson_path="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    out_path="../data/processed/kepler/04_01_model2_daily_forecast_vs_2025.geojson",
)


✅ Kepler GeoJSON written to: ../data/processed/kepler/04_01_model2_daily_forecast_vs_2025.geojson
